# Conditioned Pantun Transformer — Launcher
1. Download the repo archive from public GitHub
2. Train
3. Upload **only** the training checkpoints (`*.pt`) to Hugging Face

**Kaggle setup:** Settings → Internet **On** (needed for GitHub, tiktoken's GPT-2 BPE download and Hugging Face), Accelerator → GPU T4 x2, and add a secret named `HF_TOKEN` (write access) under Add-ons → Secrets.

In [ ]:
# ---------- CONFIG ----------
GITHUB_OWNER = "krisnaganesha1609"
GITHUB_REPO  = "Conditioned-Pantun-Transformer"
GITHUB_REF   = "main"              # branch or tag
HF_REPO_ID   = "igkganes/conditioned-pantun-transformer"
HF_PRIVATE   = False

TRAIN_ARGS = {
    "batch_size": 32,
    "grad_accum_steps": 2,         # effective batch 64
    "warmup_steps": 300,
    "max_steps": 5000,
    "eval_interval": 250,
    "eval_iters": 100,
    "lr": "6e-4",
    "min_lr": "6e-5",
}
RUN_SMOKE_TEST = True

In [ ]:
# Torch is preinstalled on Kaggle/Colab; only the extras are needed.
# (repo requirements.txt is UTF-16 with pinned versions, so it is deliberately not used)
!pip install -q tiktoken huggingface_hub

In [ ]:
# ---------- 1. Pull the archive from public GitHub ----------
import io, os, shutil, urllib.request, zipfile

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(WORK)

url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}/archive/{GITHUB_REF}.zip"
print("Downloading", url)
with urllib.request.urlopen(url, timeout=120) as r:
    data = r.read()

REPO_DIR = os.path.join(WORK, GITHUB_REPO)
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

with zipfile.ZipFile(io.BytesIO(data)) as z:
    top = z.namelist()[0].split("/")[0]        # e.g. Repo-main
    z.extractall(WORK)
os.rename(os.path.join(WORK, top), REPO_DIR)
os.chdir(REPO_DIR)
print(f"Extracted {len(data)/1e6:.1f} MB -> {REPO_DIR}")
assert os.path.exists("3.TRAINING/train.py"), "train.py not found in archive"

In [ ]:
# ---------- Hugging Face login ----------
import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            pass
assert token, "HF_TOKEN not found (Kaggle Secret, Colab secret, or env var)."
login(token=token)
print("Logged in to Hugging Face.")

In [ ]:
# ---------- 2a. Smoke test (2 steps) ----------
import subprocess, sys, shutil, os

def run(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}")

CKPT_DIR = "3.TRAINING/checkpoints"

if RUN_SMOKE_TEST:
    run([sys.executable, "3.TRAINING/train.py", "--smoke_test"])
    # the smoke test writes a 2-step last.pt; remove it so it can never be uploaded
    shutil.rmtree(CKPT_DIR, ignore_errors=True)

In [ ]:
# ---------- 2b. Full training ----------
cmd = [sys.executable, "3.TRAINING/train.py"]
for k, v in TRAIN_ARGS.items():
    cmd += [f"--{k}", str(v)]
print(" ".join(cmd))
run(cmd)

In [ ]:
# ---------- 3. Upload ONLY the checkpoints ----------
from huggingface_hub import HfApi

ckpts = sorted(f for f in os.listdir(CKPT_DIR) if f.endswith(".pt"))
assert ckpts, f"No .pt checkpoints found in {CKPT_DIR}"
print("Uploading:", ckpts)

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=HF_PRIVATE, exist_ok=True)
api.upload_folder(
    folder_path=CKPT_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model",
    path_in_repo="checkpoints",
    allow_patterns=["*.pt"],
    commit_message="Upload training checkpoints",
)
print(f"Done: https://huggingface.co/{HF_REPO_ID}/tree/main/checkpoints")